In [1]:
import xarray as xr
from pathlib import Path
import os

In [2]:
folder = Path(r"D:\Farhan\Antoine_CLASSIC_Laromaine data\Muhammad_meterological\Muhammad")

In [3]:
# Check access
print("Exists?", folder.exists())
print("Some files:", list(folder.glob("*"))[:10])

Exists? True
Some files: [WindowsPath('D:/Farhan/Antoine_CLASSIC_Laromaine data/Muhammad_meterological/Muhammad/metVar_ap_Romaine.nc'), WindowsPath('D:/Farhan/Antoine_CLASSIC_Laromaine data/Muhammad_meterological/Muhammad/metVar_lw_Romaine.nc'), WindowsPath('D:/Farhan/Antoine_CLASSIC_Laromaine data/Muhammad_meterological/Muhammad/metVar_pr_Romaine.nc'), WindowsPath('D:/Farhan/Antoine_CLASSIC_Laromaine data/Muhammad_meterological/Muhammad/metVar_qa_Romaine.nc'), WindowsPath('D:/Farhan/Antoine_CLASSIC_Laromaine data/Muhammad_meterological/Muhammad/metVar_sw_Romaine.nc'), WindowsPath('D:/Farhan/Antoine_CLASSIC_Laromaine data/Muhammad_meterological/Muhammad/metVar_ta_Romaine.nc'), WindowsPath('D:/Farhan/Antoine_CLASSIC_Laromaine data/Muhammad_meterological/Muhammad/metVar_wi_Romaine.nc'), WindowsPath('D:/Farhan/Antoine_CLASSIC_Laromaine data/Muhammad_meterological/Muhammad/Romaine_init.nc'), WindowsPath('D:/Farhan/Antoine_CLASSIC_Laromaine data/Muhammad_meterological/Muhammad/Romaine_init_

In [4]:
# (Optional) make it your working dir
os.chdir(folder)
print("CWD:", Path.cwd())

CWD: D:\Farhan\Antoine_CLASSIC_Laromaine data\Muhammad_meterological\Muhammad


In [5]:
# Examples
# List CSVs / TIFFs / NetCDFs
print("CSVs:", sorted(p.name for p in folder.glob("*.csv")))
print("TIFFs:", sorted(p.name for p in folder.glob("*.tif")))
print("NetCDFs:", sorted(p.name for p in folder.glob("*.nc")))

CSVs: []
TIFFs: []
NetCDFs: ['Romaine_init.nc', 'Romaine_init_spinedup.nc', 'Romaine_rs.nc', 'TRENDY_CO2_1700_2018.nc', 'metVar_ap_Romaine.nc', 'metVar_lw_Romaine.nc', 'metVar_pr_Romaine.nc', 'metVar_qa_Romaine.nc', 'metVar_sw_Romaine.nc', 'metVar_ta_Romaine.nc', 'metVar_wi_Romaine.nc']


In [7]:
ds = xr.open_dataset("metVar_sw_Romaine.nc")
ds

<xarray.Dataset> Size: 441MB
Dimensions:  (time: 157776, lat: 29, lon: 24)
Coordinates:
  * time     (time) float64 1MB 2.015e+07 2.015e+07 ... 2.023e+07 2.023e+07
  * lon      (lon) float32 96B -64.7 -64.6 -64.5 -64.4 ... -62.6 -62.5 -62.4
  * lat      (lat) float32 116B 53.0 52.9 52.8 52.7 52.6 ... 50.5 50.4 50.3 50.2
Data variables:
    sw       (time, lat, lon) float32 439MB ...

In [8]:
# 1) See what's inside
print(ds)                 # overview
print("Data variables:", list(ds.data_vars))  # variable names you can select
print("Coords:", list(ds.coords))

<xarray.Dataset> Size: 441MB
Dimensions:  (time: 157776, lat: 29, lon: 24)
Coordinates:
  * time     (time) float64 1MB 2.015e+07 2.015e+07 ... 2.023e+07 2.023e+07
  * lon      (lon) float32 96B -64.7 -64.6 -64.5 -64.4 ... -62.6 -62.5 -62.4
  * lat      (lat) float32 116B 53.0 52.9 52.8 52.7 52.6 ... 50.5 50.4 50.3 50.2
Data variables:
    sw       (time, lat, lon) float32 439MB ...
Data variables: ['sw']
Coords: ['time', 'lon', 'lat']


In [9]:
import numpy as np

In [10]:
VAR = list(ds.data_vars)[0]    # or set explicitly: VAR = 'ta'

In [11]:
target_lat = 50.902104
target_lon = -63.405237

In [12]:
# If file uses 0..360 longitudes, wrap the target:
if 'lon' in ds.coords:
    lon_vals = ds['lon'].values
    if np.nanmax(lon_vals) > 180:  # dataset uses 0..360
        target_lon = target_lon % 360

In [13]:
point = ds[VAR].sel(lat=target_lat, lon=target_lon, method="nearest")

In [14]:
# 3) Inspect
print(point)

<xarray.DataArray 'sw' (time: 157776)> Size: 631kB
[157776 values with dtype=float32]
Coordinates:
  * time     (time) float64 1MB 2.015e+07 2.015e+07 ... 2.023e+07 2.023e+07
    lon      float32 4B -63.4
    lat      float32 4B 50.9
Attributes:
    long_name:  Surface short-wave (solar) radiation downwards


In [15]:
# 4) If you want a time series as a tidy table
df = point.to_dataframe().reset_index()
df.head()

,time,lon,lat,sw
0,2.015010e+07,-63.400002,50.900002,0.0
1,2.015010e+07,-63.400002,50.900002,0.0
2,2.015010e+07,-63.400002,50.900002,0.0
3,2.015010e+07,-63.400002,50.900002,0.0
4,2.015010e+07,-63.400002,50.900002,0.0


In [204]:
df.to_csv("metVar_sw_Romaine.csv".format(target_lat, target_lon), index=False)

In [25]:
import xarray as xr

In [25]:
nc = r"D:\Farhan\Antoine_CLASSIC_Laromaine data\Muhammad_meterological\Muhammad\metVar_wi_Romaine.nc"  # path to your file
var = "wi"                      # change to ap, lw, qa, sw, ta, wi as needed
t0, t1 = "2015-01-01", "2023-12-31"
target_lat, target_lon = 50.90, -63.40   # <-- your point

In [26]:
# try best engine first
for eng in ("netcdf4", "h5netcdf", "scipy"):
    try:
        ds = xr.open_dataset(nc, engine=eng, decode_times=True)
        print("Opened with:", eng)
        break
    except Exception as e:
        print(f"{eng} -> {e}")

Opened with: netcdf4


In [31]:
var = "wi"                       # e.g., "ap", "lw", "qa", "sw", "ta", "wi"
target_lat, target_lon = 50.90, -63.40
t0, t1 = "2015-01-01", "2023-12-31"
out_nc = r"D:\Farhan\Antoine_CLASSIC_Laromaine data\Muhammad_meterological\Muhammad\metVar_wi2_Romaine.nc"

In [32]:
# make sure coords go in ascending order (safer for slicing)
ds = ds.sortby(["lat","lon"])

In [33]:
# time window + nearest cell
pt = ds[var].sel(time=slice(t0, t1)).sel(lat=target_lat, lon=target_lon, method="nearest")

In [34]:
# save as its own small NetCDF (1D over time, scalar lat/lon)
pt.to_dataset(name=var).to_netcdf(out_nc)
print("Wrote:", out_nc)

Wrote: D:\Farhan\Antoine_CLASSIC_Laromaine data\Muhammad_meterological\Muhammad\metVar_wi2_Romaine.nc


In [291]:
import xarray as xr

In [292]:
ds = xr.open_dataset("Romaine_init.nc")
ds

<xarray.Dataset> Size: 7MB
Dimensions:          (tile: 17, icctem: 12, lat: 17, lon: 9, layer: 20,
                      icp1: 5, ic: 4, iccp2: 14, slope: 8)
Coordinates:
  * tile             (tile) int32 68B 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17
  * layer            (layer) int32 80B 1 2 3 4 5 6 7 8 ... 14 15 16 17 18 19 20
  * ic               (ic) int32 16B 1 2 3 4
  * icp1             (icp1) int32 20B 1 2 3 4 5
  * icctem           (icctem) int32 48B 1 2 3 4 5 6 7 8 9 10 11 12
  * iccp2            (iccp2) int32 56B 1 2 3 4 5 6 7 8 9 10 11 12 13 14
  * slope            (slope) int32 32B 1 2 3 4 5 6 7 8
  * lat              (lat) float64 136B 50.4 50.5 50.6 50.7 ... 51.8 51.9 52.0
  * lon              (lon) float64 72B -63.9 -63.8 -63.7 ... -63.3 -63.2 -63.1
Data variables: (12/51)
    fcancmx          (tile, icctem, lat, lon) float64 250kB ...
    grwtheff         (tile, icctem, lat, lon) float64 250kB ...
    FARE             (tile, lat, lon) float64 21kB ...
    MID              (tile, lat, lon) float64 21kB ...
    ipeatland        (tile, lat, lon) float64 21kB ...
    SAND             (tile, layer, lat, lon) float64 416kB ...
    ...               ...
    slopefrac        (tile, slope, lat, lon) float64 166kB ...
    Cmossmas         (tile, lat, lon) float64 21kB ...
    maxAnnualActLyr  (tile, lat, lon) float64 21kB ...
    dmoss            (tile, lat, lon) float64 21kB ...
    litrmsmoss       (tile, lat, lon) float64 21kB ...
    nmtest           (lat, lon) float64 1kB ...

In [11]:
import xarray as xr
import numpy as np
from pathlib import Path

In [26]:
in_nc = r"D:\Farhan\Antoine_CLASSIC_Laromaine data\Muhammad_meterological\Muhammad\Romaine_init.nc"  # path to your file
out_nc = r"D:\Farhan\Antoine_CLASSIC_Laromaine data\Muhammad_meterological\Muhammad\Romaine_init2.nc"
target_lat, target_lon = 50.90, -63.40           # <-- your point

In [27]:
# Open (try a couple of engines)
ds = None
for eng in ("netcdf4", "h5netcdf", "scipy"):
    try:
        ds = xr.open_dataset(in_nc, engine=eng, decode_times=True)
        print(f"Opened with engine={eng}")
        break
    except Exception as e:
        last_err = e
if ds is None:
    raise RuntimeError(f"Could not open {in_nc}: {last_err}")

Opened with engine=netcdf4


In [28]:
# If your file used 'latitude'/'longitude' or 'y'/'x', normalize once:
rename = {}
if "latitude" in ds.dims:  rename["latitude"]  = "lat"
if "longitude" in ds.dims: rename["longitude"] = "lon"
if rename:
    ds = ds.rename(rename)

In [29]:
# Safety: make coords monotonic for nearest selection
ds = ds.sortby(["lat", "lon"])

# Find the nearest existing coordinates
lat_sel = float(ds["lat"].sel(lat=target_lat, method="nearest"))
lon_sel = float(ds["lon"].sel(lon=target_lon, method="nearest"))
print(f"Nearest cell -> lat={lat_sel:.4f}, lon={lon_sel:.4f}")

Nearest cell -> lat=50.9000, lon=-63.4000


In [30]:
# IMPORTANT: select with lists so lat/lon remain real dims of size 1
pt = ds.sel(lat=[lat_sel], lon=[lon_sel])

In [31]:
# Ensure **every** variable actually carries (lat, lon) dims (size=1).
# (Some variables may be only over tile/layer/etc.; we broadcast them.)
for v in list(pt.data_vars):
    if ("lat" not in pt[v].dims) or ("lon" not in pt[v].dims):
        pt[v] = pt[v].expand_dims(lat=pt.lat, lon=pt.lon)

In [32]:
# Optional: order dims nicely: (time, tile, layer, lat, lon, others…)
for v in list(pt.data_vars):
    dims = list(pt[v].dims)
    ordered = []
    for d in ("time", "tile", "layer", "lat", "lon"):
        if d in dims:
            ordered.append(d)
    ordered += [d for d in dims if d not in ordered]
    pt[v] = pt[v].transpose(*ordered)

In [33]:
# --- write output ---
try:
    pt.to_netcdf(out_nc, engine="netcdf4")
except Exception:
    pt.to_netcdf(out_nc, engine="scipy")  # NetCDF-3 fallback

print("Wrote:", out_nc)
print("New sizes:", dict(pt.sizes))
# Quick check that no variable still has 'tile'
for v in list(pt.data_vars)[:10]:
    assert "tile" not in pt[v].dims, f"{v} still has 'tile' dim!"

Wrote: D:\Farhan\Antoine_CLASSIC_Laromaine data\Muhammad_meterological\Muhammad\Romaine_init2.nc
New sizes: {'tile': 17, 'lat': 1, 'lon': 1, 'icctem': 12, 'layer': 20, 'icp1': 5, 'ic': 4, 'iccp2': 14, 'slope': 8}


AssertionError: fcancmx still has 'tile' dim!

In [34]:
import xarray as xr

In [24]:
ds = xr.open_dataset("Romaine_rs2.nc")
ds

<xarray.Dataset> Size: 45kB
Dimensions:          (tile: 17, lat: 1, lon: 1, icctem: 12, layer: 20, icp1: 5,
                      ic: 4, iccp2: 14, slope: 8)
Coordinates:
  * tile             (tile) int32 68B 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17
  * layer            (layer) int32 80B 1 2 3 4 5 6 7 8 ... 14 15 16 17 18 19 20
  * ic               (ic) int32 16B 1 2 3 4
  * icp1             (icp1) int32 20B 1 2 3 4 5
  * icctem           (icctem) int32 48B 1 2 3 4 5 6 7 8 9 10 11 12
  * iccp2            (iccp2) int32 56B 1 2 3 4 5 6 7 8 9 10 11 12 13 14
  * slope            (slope) int32 32B 1 2 3 4 5 6 7 8
  * lat              (lat) float64 8B 50.9
  * lon              (lon) float64 8B -63.4
Data variables: (12/51)
    fcancmx          (tile, lat, lon, icctem) float64 2kB ...
    grwtheff         (tile, lat, lon, icctem) float64 2kB ...
    FARE             (tile, lat, lon) float64 136B ...
    MID              (tile, lat, lon) float64 136B ...
    ipeatland        (tile, lat, lon) float64 136B ...
    SAND             (tile, layer, lat, lon) float64 3kB ...
    ...               ...
    slopefrac        (tile, lat, lon, slope) float64 1kB ...
    Cmossmas         (tile, lat, lon) float64 136B ...
    maxAnnualActLyr  (tile, lat, lon) float64 136B ...
    dmoss            (tile, lat, lon) float64 136B ...
    litrmsmoss       (tile, lat, lon) float64 136B ...
    nmtest           (lat, lon) float64 8B ...
Attributes:
    row_bounds:  6 6

In [25]:
ds.fcancmx

<xarray.DataArray 'fcancmx' (tile: 17, lat: 1, lon: 1, icctem: 12)> Size: 2kB
[204 values with dtype=float64]
Coordinates:
  * tile     (tile) int32 68B 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17
  * icctem   (icctem) int32 48B 1 2 3 4 5 6 7 8 9 10 11 12
  * lat      (lat) float64 8B 50.9
  * lon      (lon) float64 8B -63.4
Attributes:
    long_name:  Fractional coverage of CTEM PFTs per grid cell
    units:      [ ]
    _dtype:     float